# Embedding-Based Movie Recommendation System

## Objective

In this notebook, we build a personalized movie recommendation system using **Matrix Factorization (Truncated SVD)** and **FAISS Vector Search**.

Unlike traditional collaborative filtering, this approach learns dense vector representations (embeddings) of users. Users with similar preferences are located efficiently using FAISS, and their highly-rated movies are used to generate personalized recommendations.

---

## Workflow

1. Load processed dataset
2. Load learned user embeddings
3. Load user mapping
4. Load FAISS index
5. Search nearest users
6. Collect neighbor preferences
7. Remove watched movies
8. Rank candidate movies
9. Display Top-N recommendations
10. Save recommendation results

---

## Learning Outcomes

After completing this notebook, you will understand:

- Matrix Factorization based recommendation
- Dense user embeddings
- Approximate Nearest Neighbor Search using FAISS
- Personalized recommendation generation
- Industrial recommendation pipeline

In [1]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

import faiss

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


## Load Saved Files

The previous notebook generated several files that will be used here.

These include:

- User embeddings
- User mapping
- Processed dataset
- FAISS index

These files allow us to avoid retraining the model and directly generate recommendations.

In [4]:
import pandas as pd

# Ratings
ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

# Movies
movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    usecols=[0, 1],
    names=["movie_id", "movie_title"]
)

# Merge
movie_ratings = pd.merge(
    ratings,
    movies,
    on="movie_id"
)

print("Files Loaded Successfully")
movie_ratings.head()

Files Loaded Successfully


,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [5]:
import faiss
import pandas as pd

index = faiss.read_index("models/user_faiss.index")

user_embeddings = pd.read_csv("models/user_embeddings.csv")

user_mapping = pd.read_csv("models/user_mapping.csv")

print("Everything Loaded Successfully!")

Everything Loaded Successfully!


In [6]:
# ==========================================
# Search Similar Users using FAISS
# ==========================================

query_user = 1          # Original User ID
top_k = 5

# Convert Original User ID → Encoded User ID
query_encoded = user_mapping[
    user_mapping["user_id"] == query_user
].index[0]

# Query Vector
query_vector = user_embeddings.iloc[
    query_encoded
].values.astype("float32").reshape(1, -1)

# Search
distances, indices = index.search(query_vector, top_k + 1)

print("Query User :", query_user)
print()

print("Top Similar Users")
print("-"*40)

for rank in range(1, top_k + 1):

    encoded_user = indices[0][rank]

    original_user = user_mapping.iloc[
        encoded_user
    ]["user_id"]

    print(f"{rank}. User {original_user}")
    print(f"Distance : {distances[0][rank]:.2f}")
    print("-"*40)

Query User : 1

Top Similar Users
----------------------------------------
1. User 514
Distance : 1013.29
----------------------------------------
2. User 268
Distance : 1053.23
----------------------------------------
3. User 868
Distance : 1076.09
----------------------------------------
4. User 198
Distance : 1095.61
----------------------------------------
5. User 64
Distance : 1100.30
----------------------------------------


# Search Similar Users using FAISS

In this section, we retrieve the nearest users for a given query user using the FAISS index.

## Steps

- Select a query user.
- Convert the original user ID to its encoded index.
- Retrieve the corresponding embedding vector.
- Search the FAISS index.
- Return the Top-K most similar users.

FAISS performs this search in milliseconds, making it suitable for large-scale recommendation systems.

# Generate Movie Recommendations using Similar Users

Finding similar users is only the first step.

The actual recommendation process consists of:

1. Find the Top-K similar users using FAISS.
2. Retrieve the movies liked by those users.
3. Remove the movies already watched by the query user.
4. Count how frequently each movie appears among similar users.
5. Recommend the highest-ranked unseen movies.

This approach combines Matrix Factorization (SVD) with Vector Search (FAISS), allowing fast and scalable personalized recommendations.

In [7]:
# ==========================================
# Movies Already Watched by Query User
# ==========================================

watched_movies = set(
    movie_ratings[
        movie_ratings["user_id"] == query_user
    ]["movie_title"]
)

print("Movies Already Watched :", len(watched_movies))

list(watched_movies)[:10]

Movies Already Watched : 271


['Godfather: Part II, The (1974)',
 'Kansas City (1996)',
 'Mighty Aphrodite (1995)',
 'Ridicule (1996)',
 'Free Willy (1993)',
 'Haunted World of Edward D. Wood Jr., The (1995)',
 'Indiana Jones and the Last Crusade (1989)',
 'Richard III (1995)',
 'Kids in the Hall: Brain Candy (1996)',
 'Jean de Florette (1986)']

In [8]:
# ==========================================
# Collect Movies from Similar Users
# ==========================================

recommended_movies = []

for encoded_user in indices[0][1:]:      # Skip query user itself

    original_user = user_mapping.iloc[
        encoded_user
    ]["user_id"]

    movies = movie_ratings[
        movie_ratings["user_id"] == original_user
    ]

    # Keep only highly rated movies
    movies = movies[
        movies["rating"] >= 4
    ]

    recommended_movies.extend(
        movies["movie_title"].tolist()
    )

print("Collected Movies :", len(recommended_movies))

Collected Movies : 511


## Why Only Ratings ≥ 4?

Not every movie watched by similar users should be recommended.

We only keep movies rated **4 or 5 stars**, assuming they represent movies the user genuinely liked.

This filtering improves recommendation quality by removing average or disliked movies.

In [9]:
# ==========================================
# Remove Already Watched Movies
# ==========================================

recommended_movies = [
    movie
    for movie in recommended_movies
    if movie not in watched_movies
]

print("Remaining Movies :", len(recommended_movies))

Remaining Movies : 198


In [10]:
# ==========================================
# Rank Recommendations
# ==========================================

from collections import Counter

movie_counts = Counter(recommended_movies)

top_movies = movie_counts.most_common(10)

recommendations = pd.DataFrame(
    top_movies,
    columns=[
        "Movie",
        "Score"
    ]
)

recommendations

,Movie,Score
0,Stand by Me (1986),5
1,Glory (1989),4
2,Gandhi (1982),4
3,Dr. Strangelove or: How I Learned to Stop Worr...,4
4,Trainspotting (1996),4
5,"Day the Earth Stood Still, The (1951)",3
6,Real Genius (1985),3
7,Schindler's List (1993),3
8,"Close Shave, A (1995)",3
9,One Flew Over the Cuckoo's Nest (1975),3


# Results and Discussion

The recommendation engine successfully generated personalized movie recommendations using FAISS Vector Search.

## Workflow

1. Retrieved the nearest users using FAISS.
2. Collected highly rated movies (rating ≥ 4) from similar users.
3. Removed movies already watched by the query user.
4. Ranked the remaining movies based on recommendation frequency.

## Results

- Query User: 1
- Movies Already Watched: 271
- Movies Collected from Similar Users: 511
- Final Candidate Movies: 198
- Top Recommendations Generated: 10

## Conclusion

This notebook demonstrates how Matrix Factorization (SVD) combined with FAISS enables efficient nearest-neighbor search for recommendation systems.

Unlike traditional collaborative filtering, FAISS provides extremely fast similarity search, making it suitable for large-scale recommendation applications with millions of users.

The final recommendation pipeline is:

Original Ratings
→ Matrix Factorization (SVD)
→ User Embeddings
→ FAISS Vector Search
→ Similar Users
→ Personalized Movie Recommendations